## Modelo preditivo LSTM - Tech Challenge Fase 04

### Instalação e importação de bibliotecas

In [1]:
%pip install -q yfinance tensorflow

Note: you may need to restart the kernel to use updated packages.


In [22]:
import yfinance as yf
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import MeanAbsoluteError

import keras_tuner as kt

import plotly.graph_objects as go

import joblib

### Carregamento dos dados

In [2]:
acao = 'WEGE3.SA'
data_inicio = '2018-01-01'
data_fim = '2025-12-31'

dados = yf.download(acao, start = data_inicio, end = data_fim)

[*********************100%***********************]  1 of 1 completed


In [3]:
dados = dados.stack(level='Ticker').reset_index()
dados

Price,Date,Ticker,Close,High,Low,Open,Volume
0,2018-01-02,WEGE3.SA,8.220531,8.269814,8.000396,8.000396,4812860
1,2018-01-03,WEGE3.SA,8.095679,8.197532,8.079251,8.181105,4652960
2,2018-01-04,WEGE3.SA,8.016826,8.187676,8.016826,8.141678,3317600
3,2018-01-05,WEGE3.SA,8.049679,8.098964,7.980682,8.089106,2552680
4,2018-01-08,WEGE3.SA,8.115387,8.164670,7.905110,8.052961,3346200
...,...,...,...,...,...,...,...
1983,2025-12-22,WEGE3.SA,47.660000,48.849998,47.180000,48.700001,6391700
1984,2025-12-23,WEGE3.SA,48.279999,48.279999,47.720001,47.810001,4886100
1985,2025-12-26,WEGE3.SA,48.750000,48.750000,47.980000,48.020000,1829300
1986,2025-12-29,WEGE3.SA,48.709999,49.060001,48.270000,48.990002,3845900


In [4]:
dados.columns.name = None
dados['Date'] = pd.to_datetime(dados['Date'], format='%Y/%m/%d')

In [5]:
dados

,Date,Ticker,Close,High,Low,Open,Volume
0,2018-01-02,WEGE3.SA,8.220531,8.269814,8.000396,8.000396,4812860
1,2018-01-03,WEGE3.SA,8.095679,8.197532,8.079251,8.181105,4652960
2,2018-01-04,WEGE3.SA,8.016826,8.187676,8.016826,8.141678,3317600
3,2018-01-05,WEGE3.SA,8.049679,8.098964,7.980682,8.089106,2552680
4,2018-01-08,WEGE3.SA,8.115387,8.164670,7.905110,8.052961,3346200
...,...,...,...,...,...,...,...
1983,2025-12-22,WEGE3.SA,47.660000,48.849998,47.180000,48.700001,6391700
1984,2025-12-23,WEGE3.SA,48.279999,48.279999,47.720001,47.810001,4886100
1985,2025-12-26,WEGE3.SA,48.750000,48.750000,47.980000,48.020000,1829300
1986,2025-12-29,WEGE3.SA,48.709999,49.060001,48.270000,48.990002,3845900


In [6]:
dados.info()

<class 'pandas.DataFrame'>
RangeIndex: 1988 entries, 0 to 1987
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype        
---  ------  --------------  -----        
 0   Date    1988 non-null   datetime64[s]
 1   Ticker  1988 non-null   str          
 2   Close   1988 non-null   float64      
 3   High    1988 non-null   float64      
 4   Low     1988 non-null   float64      
 5   Open    1988 non-null   float64      
 6   Volume  1988 non-null   int64        
dtypes: datetime64[s](1), float64(4), int64(1), str(1)
memory usage: 108.8 KB


In [7]:
dados.describe()

,Date,Close,High,Low,Open,Volume
count,1988,1988.000000,1988.000000,1988.000000,1988.000000,1.988000e+03
mean,2022-01-03 17:51:18,27.916776,28.314701,27.535443,27.917507,8.076756e+06
min,2018-01-02 00:00:00,6.534615,6.633427,6.474467,6.508836,0.000000e+00
25%,2020-01-06 18:00:00,14.916959,15.158908,14.614248,15.008555,5.138175e+06
50%,2022-01-08 12:00:00,31.331746,31.752700,30.946444,31.318198,6.972500e+06
75%,2024-01-05 18:00:00,36.258190,36.651822,35.812435,36.228281,9.907550e+06
max,2025-12-30 00:00:00,55.428947,56.640719,54.567450,55.353220,4.393400e+07
std,NaN,13.306904,13.448459,13.151438,13.297318,4.500044e+06


### Preparação dos dados

Coluna Volume está em milhões com picos extremos que podem dominar a dinâmica da LSTM. Faremos a transformação de Volume para deixar mais estacionário e reduzir impacto de outliers.

In [ ]:
dados['Volume'] = np.log1p(dados['Volume'])

Definindo características OHLCV - LSTM Multivariada

In [49]:
features = ['Open', 'High', 'Low', 'Close', 'Volume']
selecao_dados = dados[features].astype(np.float32)

Definindo a janela de observação e horizonte

In [ ]:
X_passos = 60 # janela de observação
y_passos = 1 # horizonte

Separação do treino e teste

In [50]:
qtd_dados_treino = int(0.8 * len(selecao_dados))
treino = selecao_dados.iloc[:qtd_dados_treino].values
teste  = selecao_dados.iloc[qtd_dados_treino - X_passos:].values

Normalização dos dados de treino (`fit_transform`) e de teste (`transform`) evitando vazamento de informação.

In [51]:
scaler = MinMaxScaler()

treino_scaled = scaler.fit_transform(treino)
teste_scaled = scaler.transform(teste)

Separando X e y (treino e teste) de acordo com os passos especificados

In [52]:
def criar_janela_multivariada(dados, x_passos, y_passos, alvo_index):
    X, y = [], []
    T = len(dados)

    for t in range(x_passos, T - y_passos + 1):
        X.append(dados[t - x_passos:t, :])
        y.append(dados[t:t + y_passos, alvo_index])

    return np.array(X), np.array(y)

In [53]:
alvo_index = features.index('Close')

X_treino, y_treino = criar_janela_multivariada(
    treino_scaled,
    X_passos,
    y_passos,
    alvo_index
)

X_teste, y_teste = criar_janela_multivariada(
    teste_scaled,
    X_passos,
    y_passos,
    alvo_index
)

In [54]:
print("X_treino:", X_treino.shape) # espera-se (amostra, X_passos, OHLCV)
print("y_treino:", y_treino.shape) # espera-se (amostra, y_passos)
print("X_teste :", X_teste.shape)
print("y_teste :", y_teste.shape)

X_treino: (1530, 60, 5)
y_treino: (1530, 1)
X_teste : (398, 60, 5)
y_teste : (398, 1)


### Definindo os melhores hiperparâmetros do modelo LSTM

Vamos utilizar o `keras_tuner` para encontrar os melhores hiperparâmetros para o LSTM. Buscamos equilibrar a capacidade de aprendizado e generalização em um problema de previsão multivariada de séries temporais. 

* O número de camadas ocultas é ajustado entre 1 e 3, evitando profundidade excessiva que pode introduzir overfitting em dados financeiros.
* Cada camada LSTM utiliza entre 64 e 192 unidades, permitindo capturar padrões temporais complexos oriundos das múltiplas variáveis de entrada (OHLCV).
* O uso controlado de Dropout (entre 5% e 15%) atua como regularização leve, preservando a memória temporal da rede.
* A camada de saída densa produz previsões multi‐passo, compatíveis com o horizonte definido, enquanto a otimização é realizada com o algoritmo Adam e taxas de aprendizado moderadas, visando estabilidade no treinamento e minimização do erro absoluto médio (MAE) que é uma das métricas de foco.

In [56]:
def estrutura_modelo_LSTM(hp):

    n_camadas_ocultas = hp.Choice(
        'n_hidden_layers',
        values=[1, 2, 3]
    )

    units_0 = hp.Choice(
        'units_layer_0',
        values=[64, 96, 128, 192]
    )

    modelo = Sequential()
    modelo.add(Input(shape=(X_treino.shape[1], X_treino.shape[2])))

    modelo.add(
        LSTM(
            units_0,
            return_sequences=(n_camadas_ocultas > 1)
        )
    )

    for i in range(1, n_camadas_ocultas):
        units_i = hp.Choice(
            f'units_layer_{i}',
            values=[64, 96, 128, 192]
        )

        modelo.add(
            LSTM(
                units_i,
                return_sequences=(i < n_camadas_ocultas - 1)
            )
        )

        modelo.add(
            Dropout(
                hp.Choice('dropout', [0.05, 0.1, 0.15])
            )
        )

    modelo.add(Dense(y_passos))

    modelo.compile(
        optimizer=Adam(
            hp.Choice(
                'learning_rate',
                [5e-4, 7e-4, 1e-3]
            )
        ),
        loss='mae',
        metrics=[MeanAbsoluteError(name='mae')]
    )

    return modelo

O ajuste dos hiperparâmetros foi conduzido por meio do método Hyperband (realiza uma busca eficiente no espaço de configurações ao interromper precocemente modelos com desempenho inferior). O erro absoluto médio de validação (val_mae) foi adotado como função objetivo, enquanto o número máximo de épocas foi limitado a 50 para controle do custo computacional.

In [57]:
tuner = kt.Hyperband(
    estrutura_modelo_LSTM,
    objective='val_mae',
    max_epochs=50,
    factor=3,
    directory='kt_dir',  # nome da pasta para salvar os resultados
    project_name='wege3_lstm',
    overwrite=True
)

O treinamento é encerrado após 7 épocas consecutivas sem melhoria, e os pesos do modelo são restaurados para a melhor época observada.


In [58]:
stop_early = EarlyStopping(
    monitor='val_mae',
    mode='min',
    patience=7,
    restore_best_weights=True
)

Buscando as melhores combinações de hiperparâmetros utilizando os dados de treino, reservando 10% para validação. O treinamento ocorre por até 50 épocas, com interrupção antecipada via Early Stopping, e sem embaralhamento dos dados, preservando a ordem temporal da série.

In [59]:
tuner.search(
    X_treino,
    y_treino,
    epochs=50,
    validation_split=0.1,
    callbacks=[stop_early],
    shuffle=False
)

Trial 90 Complete [00h 00m 41s]
val_mae: 0.023411087691783905

Best val_mae So Far: 0.011320414952933788
Total elapsed time: 00h 26m 21s


Coletando a melhor combinação de hiperparâmetros (`melhor_hps`) depois de testadas diversas combinações.

In [60]:
melhor_hps = tuner.get_best_hyperparameters(num_trials=3)[0]

print("Best HPs:", melhor_hps.values)

Best HPs: {'n_hidden_layers': 1, 'units_layer_0': 64, 'learning_rate': 0.001, 'units_layer_1': 96, 'dropout': 0.15, 'units_layer_2': 96, 'tuner/epochs': 50, 'tuner/initial_epoch': 17, 'tuner/bracket': 3, 'tuner/round': 3, 'tuner/trial_id': '0047'}


### Treinamento do modelo LSTM

In [61]:
modelo_lstm = tuner.hypermodel.build(melhor_hps)

modelo_lstm.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 64)             │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,985 (70.25 KB)

 Trainable params: 17,985 (70.25 KB)

 Non-trainable params: 0 (0.00 B)

In [63]:
modelo_lstm.fit(X_treino, y_treino, epochs=100, validation_split= 0.1, callbacks=[stop_early])

Epoch 1/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0201 - mae: 0.0201 - val_loss: 0.0344 - val_mae: 0.0344
Epoch 2/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0196 - mae: 0.0196 - val_loss: 0.0131 - val_mae: 0.0131
Epoch 3/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0170 - mae: 0.0170 - val_loss: 0.0143 - val_mae: 0.0143
Epoch 4/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0163 - mae: 0.0163 - val_loss: 0.0126 - val_mae: 0.0126
Epoch 5/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0160 - mae: 0.0160 - val_loss: 0.0209 - val_mae: 0.0209
Epoch 6/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0157 - mae: 0.0157 - val_loss: 0.0124 - val_mae: 0.0124
Epoch 7/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0158 - mae: 0.0158 - val_loss: 0.0129 - val_mae: 0.0129
Epoch 8/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0159 - mae: 0.0159 - val_loss: 0.0197 - val_mae: 0.0197
Epoch 9/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - lo

### Avaliação do modelo

Primeiro é preciso inverter a transformação do `scaler`

In [ ]:
def inverter_transformacao(
    valores_normalizados,
    scaler=scaler,
    num_features=len(features),
    alvo_index=alvo_index,
    y_passos=y_passos):

    valores_flat = valores_normalizados.reshape(-1)

    dummy = np.zeros((len(valores_flat), num_features))
    dummy[:, alvo_index] = valores_flat

    valores_desnormalizados = scaler.inverse_transform(dummy)[:, alvo_index]

    return valores_desnormalizados.reshape(-1, y_passos)

In [ ]:
predictions = modelo_lstm.predict(X_teste)
preds_desnormalizado = inverter_transformacao(predictions)

y_teste_desnormalizado = inverter_transformacao(y_teste)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


Observando as métricas MAE, RMSE e MAPE.

In [98]:
mae = mean_absolute_error(y_teste_desnormalizado, preds_desnormalizado)
rmse = np.sqrt(mean_squared_error(y_teste_desnormalizado, preds_desnormalizado))
mape = np.mean(np.abs((y_teste_desnormalizado - preds_desnormalizado) / y_teste_desnormalizado)) * 100

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAPE: {mape:.2f}%")

MAE:  0.7331
RMSE: 1.0216
MAPE: 1.61%


### Visualização dos dados

In [117]:
y_plot = y_teste_desnormalizado.flatten()
pred_plot = preds_desnormalizado.flatten()

dates = dados['Date'].iloc[qtd_dados_treino:]

df_plot = pd.DataFrame({
    'Date': dates.values,
    'Real': y_plot,
    'Previsto': pred_plot
})

In [118]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot['Date'],
    y=df_plot['Real'],
    mode='lines',
    name='Preço Real',
    line=dict(width=2)
))

fig.add_trace(go.Scatter(
    x=df_plot['Date'],
    y=df_plot['Previsto'],
    mode='lines',
    name='Preço Previsto',
    line=dict(width=2, dash='dash')
))

fig.update_layout(
    title='Preço Real vs Preço Previsto (LSTM)',
    xaxis_title='Data',
    yaxis_title='Preço de Fechamento',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

In [ ]:
y_treino_pred = modelo_lstm.predict(X_treino)

preds_treino_desnormalizado = inverter_transformacao(y_treino_pred)
y_treino_desnormalizado = inverter_transformacao(y_treino)

48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [ ]:
y_plot_treino = y_treino_desnormalizado.flatten()
pred_plot_treino = preds_treino_desnormalizado.flatten()

datas_treino = dados['Date'].iloc[X_passos:qtd_dados_treino]

In [119]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=datas_treino,
    y=y_plot_treino,
    mode='lines',
    name='Treino - Real'
))

fig.add_trace(go.Scatter(
    x=datas_treino,
    y=pred_plot_treino,
    mode='lines',
    name='Treino - Previsto',
    line=dict(dash='dot')
))

fig.add_trace(go.Scatter(
    x=df_plot['Date'],
    y=df_plot['Real'],
    mode='lines',
    name='Validação - Real'
))

fig.add_trace(go.Scatter(
    x=df_plot['Date'],
    y=df_plot['Previsto'],
    mode='lines',
    name='Validação - Previsto',
    line=dict(dash='dash')
))

fig.update_layout(
    title='Treino e Validação — Real vs Previsto (LSTM)',
    xaxis_title='Data',
    yaxis_title='Preço de Fechamento',
    template='plotly_white',
    hovermode='x unified'
)

fig.show()


In [120]:
df_plot['Erro'] = df_plot['Real'] - df_plot['Previsto']

In [121]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot['Date'],
    y=df_plot['Erro'],
    mode='lines',
    name='Erro'
))

fig.update_layout(
    title='Erro da Previsão ao Longo do Tempo',
    xaxis_title='Data',
    yaxis_title='Erro (Real - Previsto)',
    template='plotly_white'
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="red",
    line_width=4,
    annotation_text="Linha de erro nulo",
    annotation_position="top left"
)

fig.show()


### Salvar o modelo

In [127]:
modelo_lstm.save('../models/modelo_lstm.keras')

In [128]:
joblib.dump(scaler, '../models/scaler.joblib')

['../models/scaler.joblib']